# HRAI API demo notebook

## Spuštění API a notebooku současně

1. V jednom terminálu spusťte API server:
   ```bash
   uvicorn main:app
   ```
   Server poběží na `http://127.0.0.1:8000`.

2. V dalším terminálu spusťte Jupyter notebook v prohlížeči (ne v IDE — může dojít ke konfliktu portů):
   ```bash
   jupyter notebook --no-browser
   ```
   Poté notebook otevřete v prohlížeči na URL, kterou Jupyter vypíše.

3. Spouštějte buňky postupně.

---

In [ ]:
!pip install --upgrade pip
!pip install -r requirements.txt

In [1]:
import os, json, random, textwrap, requests
from pathlib import Path

from pos_extraction import text_to_ngrams
from query import extract_from_resume
from dotenv import load_dotenv

from rich import print as rprint

In [ ]:
# pro rychlejší fungování encoderu doporučuji nastavit HF_TOKEN:
os.environ['HF_TOKEN'] = ''

In [2]:
load_dotenv()

BASE_URL = 'http://127.0.0.1:8000'

## Ukázky

In [3]:
resumes = json.loads(Path(os.path.join('data', 'cs_resumes.json')).read_text(encoding='utf-8'))
text = random.choice(resumes)

print(textwrap.shorten(text, width=600, placeholder='...'))

Odborník na identifikaci a dokumentaci případů zneužívání dětí. Umí najít a implementovat nejlepší možná řešení. Přednosti Licence na ochranu dětí Behaviorální terapie Empatický zdravý úsudek Vášeň pro sociální práci Obeznámenost se soudními postupy Úspěchy Zvládl více než padesát až čtyřicet pět klientů v daném okamžiku. Zkušenosti na Současný Family Advocate Název společnosti - Město, Stát Nábor ve čtvrtích, které jsou v blízkosti center Head Start / Early Head Start, které jsou pod úrovní chudoby, v agenturách sociálních služeb, útulcích pro bezdomovce, zdrojových akcích pro děti a...


#### vytvoření frází

In [4]:
ngrams = text_to_ngrams(text)
print(', '.join(sorted(ngrams)))

#1 v doporučeních, 000 zaměstnanců, 1 milion, 1 milion obchodníků, 1 provize, 10 regionu, 10 regionu experience, 100 cílů, 100 cílů proniknutí, 100 kvóty, 100 prodejních kvót, 105 tisíc dolarů, 109 škol, 109 škol jazyky, 11 měsíců, 114 týmů, 12 za čtvrtletí, 14 000 zaměstnanců, 2 b, 2 b lead, 2 místo, 200 zástupců, 200 účty, 25 místo, 3 kontejnery, 3 místě, 3 měsíce, 30 bankami, 30 doporučení, 30 účtů, 300 lidí, 35 států, 50 tisíc usd, 53 prodejů, 6 místo, 6 místě, 600 poboček, 7 měsíců, 7 měsíců mluvčí, 8 výplatních období, account executive klienti, account management, account management b, account manager, account manager účty, achievement award, akce, akce ve spolupráci, akcí, akcích, akcích 300, akcích 300 lidí, aktivací 68, b 2, b 2 b, b b, b b 2, b do b, b lead, b lead generation, b to b, b udržení, b udržení zákazníků, bakaláři, bakaláři umění, balíčky, balíčky za leden, bankami, banky, banky uzavřely, banky uzavřely partnerství, barové sady, basketbalové ligy, basketbalové lig

#### získání klíčových slov a oborů (pomocí pick_resume)

In [4]:
entities = extract_from_resume(text)
skills = [e for e in entities if e.entity_type == 'skill']
occupations = [e for e in entities if e.entity_type == 'occupation']

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
print(f"dovednosti - {len(skills)}:")
for s in skills:
    print(f'{s.label}\n'
          f'skore: {round(s.cosine_score,5)}\n'
          f'{s.description}\n')

dovednosti - 27:
ochrana dětí
skore: 0.9958
Právní předpisy a postupy k prevenci a ochraně dětí před zneužitím a újmou.

plány
skore: 0.96876
Musí být schopen číst a chápat návrhy, výkresy a plány a vést jednoduché písemné záznamy.

řídit dobrovolnické programy
skore: 0.91812
Řídit programy zaměřené na nábor, rozřazování a vysílání dobrovolníků v rámci různých úkolů a organizací na místní, vnitrostátní a mezinárodní úrovni.

soudní řízení
skore: 0.88893
Nařízení, která se uplatňují v průběhu vyšetřování věci soudem a během soudního jednání, a způsob vedení soudních řízení.

provádět nábor
skore: 0.87888
Přilákat, zobrazit, vybrat a najmout osoby, které jsou vhodné pro danou práci.

organizovat rodičovské schůzky
skore: 0.86834
Připravit společné a individuální schůzky s rodiči studentů s cílem diskutovat o akademickém pokroku jejich dítěte a celkovém prospěchu.

vládní politika
skore: 0.85919
Politické činnosti, plány a záměry vlády uplatňované v rámci legislativního zasedání ve věci k

In [5]:
print(f"\nzaměstnání - {len(occupations)}:")
for o in occupations:
     print(f'název: {o.label}\n'
          f'skore: {round(o.cosine_score,5)}\n'
          f'{o.description}\n')


zaměstnání - 0:


### Analýza životopisu s cílovým povoláním
`POST /resume` — nahraje soubor + volitelný target_job
→ `SuggestionResponse` (top_suggestion, target_suggestion)

In [7]:
resume_path = os.path.join('testfiles', 'resume.pdf')
target_job = 'frontend designer' # pokud používáš svůj soubor, můžeš skecifikovat práci pro kterou životopis je

with open(resume_path, 'rb') as f:
    files = {'file': f}
    data = {'target_job': target_job}
    r = requests.post(f'{BASE_URL}/resume', files=files, data=data)

In [8]:
rprint(r.json())

{
    'top_suggestion': {
        'occupation': {
            'id': 'http://data.europa.eu/esco/occupation/a4869ae9-2dfe-48cc-bd97-f803880f3407',
            'esco_uri': 'http://data.europa.eu/esco/occupation/a4869ae9-2dfe-48cc-bd97-f803880f3407',
            'label': 'projektant/projektantka',
            'code': '3118.3',
            'isco_code': 3118.0,
            'score': 0.7061413526535034
        },
        'missing_skills': [],
        'match_score': 0.0
    },
    'target_suggestion': {
        'occupation': {
            'id': 'http://data.europa.eu/esco/occupation/a4869ae9-2dfe-48cc-bd97-f803880f3407',
            'esco_uri': 'http://data.europa.eu/esco/occupation/a4869ae9-2dfe-48cc-bd97-f803880f3407',
            'label': 'projektant/projektantka',
            'code': '3118.3',
            'isco_code': 3118.0,
            'score': 0.7061413526535034
        },
        'missing_skills': [],
        'match_score': 0.0
    }
}

### Analýza životopisu → práce z různých oblastí pracovního trhu
`POST /resume/domains` — nahraje soubor
→ `DomainResponse` (domains: list of DomainResult, each with domain, occ_count, occupations)

In [19]:
resume_path = os.path.join('testfiles', 'resume.odt')

with open(resume_path, 'rb') as f:
    files = {'file': f}
    r = requests.post(f'{BASE_URL}/resume/domains', files=files)

In [20]:
rprint(r.json())

{'domains': []}

### Dovednosti → povolání dle skore
`POST /text` — JSON s klíčem `skills` (čárkou oddělené dovednosti) + volitelný `target_job`
→ `SkillsResponse` (suggestions: list of Suggestion)

In [21]:
payload = {
    'skills': 'Python, machine learning, analýza dat, SQL, statistika',
    'target_job': None
}
r = requests.post(f'{BASE_URL}/text', json=payload)

In [22]:
rprint(r.json())

{'suggestions': []}

### Skore zadaného povolání
`POST /text/goal` — JSON s `skills` + `target_job`
→ `TargetJobResponse` (top_suggestion, target_suggestion)

In [9]:
payload = {
    'skills': ' '.join([s.label for s in skills]),
    'target_job': 'ředitel české správy sociálního zabezpečení'
}
r = requests.post(f'{BASE_URL}/text/goal', json=payload)

In [10]:
rprint(r.json())

{
    'top_suggestion': None,
    'target_suggestion': {
        'occupation': {
            'id': 'http://data.europa.eu/esco/occupation/d00b4607-c4fe-4f3e-8974-29401e23223c',
            'esco_uri': 'http://data.europa.eu/esco/occupation/d00b4607-c4fe-4f3e-8974-29401e23223c',
            'label': 'správce sociálního zabezpečení/správkyně sociálního zabezpečení',
            'code': '1213.2.1',
            'isco_code': 1213.0,
            'score': 0.7391868233680725
        },
        'missing_skills': [],
        'match_score': 0.0
    }
}

### Vyhledávání samotných entit
`POST /query` — JSON s `text`, volitelný `entity_type` (all/occupation/skill/skill_group/isco_group), volitelný `n`
→ `List[EntityResult]`

In [19]:
query = {
    'text': 'pomocný pracovník v kychyni',
    'entity_type': 'occupation',
    'n': 5
}
r = requests.post(f'{BASE_URL}/query', json=query)

In [20]:
rprint(r.json())

[]